# COMP34812 NLI — Demo Notebook

**Group:** n  
**Category:** B — Deep Learning (non-transformer)  
**Model:** ResESIM + 1477d input embeddings

| Component | Dim | Reference |
|---|---|---|
| GloVe | 300 | Pennington et al., 2014 |
| ELMo | 1024 | Peters et al., 2018 |
| CharCNN | 100 | Kim et al., 2016 |
| POS embeddings | 50 | Universal Dependencies |
| Negation flags | 3 | spaCy dep parse + boundary |
| **Total** | **1477** | |

## Instructions

1. Run all cells top to bottom
2. Predictions saved to `Group_n_B.csv`

**Input CSV:** must have `premise` and `hypothesis` columns

```
test.csv
    ↓  ELMo pre-computation  (~5 mins, A100)
    ↓  GloVe + CharCNN + POS + Negation
    ↓  inference_embeddings.npz  [N, 64, 1477]
    ↓  OracleNet (ResESIM)
    ↓  Group_n_B.csv
```

## 1. Environment Setup

**Skip this cell if running locally** — only needed on Colab.

## 2. Setup Path + Verify Files

In [35]:
import sys
sys.path.append('.')
!{sys.executable} -m pip install -q spacy
!{sys.executable} -m spacy download en_core_web_sm
!{sys.executable} -m pip install scikit-learn
print('spaCy model ready.')
print('Repo ready.')




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 26.4 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
spaCy model ready.
Repo ready.


In [36]:
# ── Verify required files ─────────────────────────────────────────────────────
import os

required = {
    './notebook_data/meta.pt': 'Vocab + GloVe meta',
    './final_model_versions/ff2f02d4/best_model.pt': 'Trained model weights',
    './data/test.csv': 'Test data for inference',
    './notebook_data/elmo_model/options.json': 'ELMo options',
    './notebook_data/elmo_model/weights.hdf5': 'ELMo weights',
}

all_ok = True
for path, desc in required.items():
    exists = os.path.exists(path)
    size   = f'  ({os.path.getsize(path)/1e6:.1f} MB)' if exists else ''
    status = '✓' if exists else '✗  MISSING'
    print(f'  {status}  {path}{size}')
    if not exists: all_ok = False

print()
print('All files present — continue.' if all_ok else 'Fix missing files before continuing.')

  ✓  ./notebook_data/meta.pt  (1.0 MB)
  ✓  ./final_model_versions/ff2f02d4/best_model.pt  (86.9 MB)
  ✓  ./data/test.csv  (0.4 MB)
  ✓  ./notebook_data/elmo_model/options.json  (0.0 MB)
  ✓  ./notebook_data/elmo_model/weights.hdf5  (374.4 MB)

All files present — continue.


## 3. Pre-compute Embeddings

Uses `EmbeddingPrecomputer` from `precomputeClasses.py`.  
Steps run automatically:
1. Load meta (vocab, char2idx, pos2idx, GloVe)
2. Build embedding layers (CharCNN, POS, GloVe)
3. ELMo pre-computation via Python 3.10 venv
4. Full 1477d pre-computation
5. Save `inference_embeddings.npz`

In [ ]:
from precomputeClasses import EmbeddingPrecomputer

pc = EmbeddingPrecomputer(
    meta_path    = './notebook_data/meta.pt',
    elmo_options = './notebook_data/elmo_model/options.json',
    elmo_weights = './notebook_data/elmo_model/weights.hdf5',
    elmo_venv    = './venv310/bin/python',
)

pc.run(
    csv_path   = './data/test.csv',
    output_npz = './notebook_data/inference_embeddings.npz',
)

## 4. Load Model + Run Inference

In [38]:
# ── Select device ─────────────────────────────────────────────────────────────
import torch

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device: {device}')

Device: mps


In [39]:
# ── Load dataset and dataloader ────────────────────────────────────────────────
from res_esim.loader.res_esim_dataset import ResESIM_Dataset
from torch.utils.data import DataLoader

dataset = ResESIM_Dataset('./notebook_data/inference_embeddings.npz')
loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

Loading ./notebook_data/inference_embeddings.npz...
  50 samples, dim=1477


In [40]:
# ── Load OracleNet model ──────────────────────────────────────────────────────
from res_esim.model_layers.oracle_net import OracleNet

model = OracleNet(
    input_dim=1477,
    hidden_dim=512,
    num_blocks=3,
    num_classes=2,
    dropout_rate=0.2,
    num_heads=8,
)
model.load_state_dict(torch.load(
    './final_model_versions/ff2f02d4/best_model.pt',
    map_location=device,
    weights_only=False,
))
model.to(device).eval()
print('Model loaded and set to eval mode.')

Model loaded and set to eval mode.


In [41]:
# ── Run inference ─────────────────────────────────────────────────────────────
from tqdm import tqdm

all_preds = []
with torch.no_grad():
    for batch in tqdm(loader, desc='inference'):
        logits = model(
            batch['premise_embedding'].to(device),
            batch['hyp_embedding'].to(device),
            batch['premise_length'].to(device),
            batch['hyp_length'].to(device),
        )
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())

print(f'Inference complete: {len(all_preds)} predictions')

inference: 100%|██████████| 2/2 [00:00<00:00,  6.23it/s]

Inference complete: 50 predictions


In [ ]:
# ── Save predictions ──────────────────────────────────────────────────────────
import pandas as pd

pd.DataFrame({'prediction': all_preds}).to_csv('Group_n_B.csv', index=False)

print(f'Saved Group_n_B.csv  ({len(all_preds)} predictions)')
print(f'Label distribution: {pd.Series(all_preds).value_counts().sort_index().to_dict()}')

## Done

Predictions saved to `Group_n_B.csv`  
Download and submit via Canvas.